In [ ]:

from typing import TypedDict

class AgentState(TypedDict):
    question: str
    documents: list
    answer: str
    retry_count: int


In [ ]:

def retrieve_documents(state):
    docs = search_vector_store(state["question"])
    return {"documents": docs}

def generate_answer(state):
    answer = call_llm(state["question"], state["documents"])
    return {"answer": answer}


In [3]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(AgentState)
graph.add_node("retrieve", retrieve_documents)
graph.add_node("generate", generate_answer)

graph.add_edge(START, "retrieve")
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", END)


In [ ]:
def should_retry(state):
    if state["answer"] == "" and state["retry_count"] < 3:
        return "retrieve"    # Loop back
    return END               # Done

graph.add_conditional_edges("generate", should_retry)